In [19]:
import numpy as np
import joblib

import pandas as pd
pd.set_option("display.max_columns", None)


In [ ]:
dados = pd.read_csv('../data/klike_challenge_dataset.csv')
dados.head(5)

In [ ]:
df_hook = dados.groupby(['platform', 'has_hook'], observed=True)['roas'].agg(['count', 'median'])
com_hook = df_hook.xs(True, level='has_hook')['median']
sem_hook = df_hook.xs(False, level='has_hook')['median']

diferenca_pct = ((com_hook - sem_hook) / sem_hook) * 100
display(diferenca_pct.round(2))
df_hook

In [27]:
# Funções ----------------------------------------------------------------------------------------------------------------------

def comparar_atributo(dados, atributo, metrica, atual, alternativa):
    df = dados.groupby(['platform', atributo], observed=True)[metrica].agg(['count', 'median'])
    grupo_atual = df.xs(atual, level=atributo)['median']
    grupo_alternativa = df.xs(alternativa, level=atributo)['median']
    diferenca_pct = ((grupo_alternativa - grupo_atual) / grupo_atual) * 100
    return diferenca_pct

def melhor_alternativa(dados, plataforma, atributo, metrica, valor_atual):
    df = dados[dados['platform'] == plataforma]
    resumo = df.groupby(atributo, observed=True)[metrica].agg(['count', 'median'])
    # Considera só grupos com pelo menos 10 campanhas
    resumo = resumo[resumo['count'] >= 10]

    if valor_atual not in resumo.index:
        return None

    mediana_atual = resumo.loc[valor_atual, 'median']
    # Remove a categoria atual das alternativas
    alternativas = resumo.drop(index=valor_atual)

    if alternativas.empty or mediana_atual <= 0:
        return None

    # Seleciona alternativa com maior mediana
    melhor_valor = alternativas['median'].idxmax()
    melhor_mediana = alternativas.loc[melhor_valor, 'median']

    diferenca_pct = (melhor_mediana - mediana_atual) / mediana_atual * 100

    if isinstance(valor_atual, bool) and isinstance(melhor_valor, bool):
        valor_atual = bool(valor_atual)
        melhor_valor = bool(melhor_valor)

    if diferenca_pct <= 0:
        return None

    return {'atributo': atributo, 'valor_atual': valor_atual, 'valor_recomendado': melhor_valor, 
            'metrica': metrica, 'diferenca_pct': float(diferenca_pct), 'mediana_atual': float(mediana_atual),
            'mediana_alternativa': float(melhor_mediana), 'n_atual': int(resumo.loc[valor_atual, 'count']),
            'n_alternativa': int(resumo.loc[melhor_valor, 'count'])}

def metrica_por_objetivo(objetivo):
    metricas_por_objetivo = {
    "traffic": "ctr",
    "engagement": "engagement_rate",
    "conversions": "roas",
    "awareness": "watch_time_ratio",
    "app_install": "conversion_rate"}
    return metricas_por_objetivo[objetivo]

def gerar_candidatas(campanha, dados):
    candidatas = []
    # Identificar a métrica principal da campanha
    metrica_principal = metrica_por_objetivo(campanha['objective'])
    metricas = ['ctr', 'conversion_rate', 'roas', 'watch_time_ratio', 'engagement_rate']
    metricas_ordenadas = [metrica_principal] + [m for m in metricas if m != metrica_principal]

    # Percorrer métricas e atributos modificáveis
    atributos = ["has_hook", "has_cta", "has_face", "has_subtitle", "text_density", "duration_group", "audio_group", "format"]
    for atributo in atributos:
        if pd.isna(campanha[atributo]):
            continue
        for metrica in metricas_ordenadas:
            resultado = melhor_alternativa(dados=dados, plataforma=campanha['platform'], atributo=atributo, metrica=metrica, 
                                                         valor_atual=campanha[atributo])
            if resultado is not None:
                resultado["metrica_principal"] = (metrica == metrica_principal)
                candidatas.append(resultado)
    
    return candidatas

def priorizar_recomendacoes(candidatas, max_recs=5, diferenca_minima=10):
    recs_finais = []
    
    recs_metrica_principal = [rec for rec in candidatas if rec["metrica_principal"]]
    recs_metrica_principal = sorted(recs_metrica_principal, key=lambda rec: rec["diferenca_pct"], reverse=True)
    
    recs_outras = [rec for rec in candidatas if not rec["metrica_principal"]]
    recs_outras = sorted(recs_outras, key=lambda rec: rec["diferenca_pct"], reverse=True)

    atributos_selecionados = set()

    for rec in recs_metrica_principal:
        if len(recs_finais) >= min(2, max_recs):
            break

        if rec["diferenca_pct"] >= diferenca_minima and rec["atributo"] not in atributos_selecionados:
            recs_finais.append(rec)
            atributos_selecionados.add(rec["atributo"])

    for rec in recs_outras:
        if len(recs_finais) >= max_recs:
            break

        if rec["diferenca_pct"] >= diferenca_minima and rec["atributo"] not in atributos_selecionados:
            recs_finais.append(rec)
            atributos_selecionados.add(rec["atributo"])

    recs_finais = sorted(recs_finais, key=lambda rec: rec["diferenca_pct"], reverse=True)
    return recs_finais

def aplicar_recomendacoes(campanha, recomendacoes, dados):
    campanha_modificada = campanha.copy()
    for rec in recomendacoes:
        atributo = rec["atributo"]

        if atributo in ["duration_group", "audio_group"]:
            coluna_original = {"duration_group": "video_duration_s", "audio_group": "music_voice_ratio"}[atributo]
            valor_recomendado = dados.loc[(dados["platform"] == campanha["platform"]) & (dados[atributo] == rec["valor_recomendado"]), 
                                          coluna_original].median()
            campanha_modificada[coluna_original] = float(valor_recomendado)
        else:
            campanha_modificada[atributo] = rec["valor_recomendado"]
            
    return campanha_modificada

def avaliar_score_candidatas(camapanha, candidatas, dados, preprocessor, ridge, features):
    candidatas_avaliadas = []

    X_original = camapanha.to_frame().T[features]
    score_original = ridge.predict(preprocessor.transform(X_original))[0]

    for rec in candidatas:
        campanha_modificada = aplicar_recomendacoes(camapanha, [rec], dados)
        X_modificada = campanha_modificada.to_frame().T[features]
        
        score_modificado = ridge.predict(preprocessor.transform(X_modificada))[0]

        rec_avaliada = rec.copy()
        rec_avaliada["delta_score"] = float(score_modificado - score_original)

        candidatas_avaliadas.append(rec_avaliada)

    return candidatas_avaliadas

In [ ]:
# Criando grupos de duração para tratar como categórica

dados['duration_group'] = pd.qcut(dados['video_duration_s'], q=3, labels=['short', 'medium', 'long'])

df_duration = dados.groupby(['platform', 'duration_group'], observed=True)['engagement_rate'].agg(['count', 'median'])
display(df_duration)

In [ ]:
# Criando grupos de audio para tratar como categórica

dados['audio_group'] = pd.qcut(dados['music_voice_ratio'], q=3, labels=['low_music', 'medium_music', 'high_music'])

df_audio = dados.groupby(['platform', 'audio_group'], observed=True)['engagement_rate'].agg(['count', 'median'])
display(df_audio)

dados['music_voice_ratio'].quantile([0, 1/3, 2/3, 1])

In [ ]:
# Criando taxa de conversão com base na quantidade de cliques

dados["conversion_rate"] = (dados["conversions"] / dados["clicks"].replace(0, np.nan))

display(dados["conversion_rate"].describe())
print("Valores ausentes:", dados["conversion_rate"].isna().sum())
print("Taxas acima de 100%:", (dados["conversion_rate"] > 1).sum())

In [ ]:
print(dados[["avg_watch_time_s", "video_duration_s"]].isna().sum())

print("Durações iguais a zero:",
      (dados["video_duration_s"] == 0).sum())

In [ ]:
# Criando a proporção do vídeo que foi assistida

dados["watch_time_ratio"] = dados["avg_watch_time_s"] / dados["video_duration_s"].replace(0, np.nan)

display(dados["watch_time_ratio"].describe())
print("Valores ausentes:", dados["watch_time_ratio"].isna().sum())
print("Proporções acima de 100%:", (dados["watch_time_ratio"] > 1).sum())

In [20]:
# Calcular o novo klike_score
ferramentas = joblib.load("../models/ridge_model.joblib")

preprocessor = ferramentas["preprocessor"]
ridge = ferramentas["model"]

In [ ]:
# CÉLULA DE TESTE --------------------------------------------------------------------------------------------
campanha = dados.iloc[4]

candidatas = gerar_candidatas(campanha, dados)
recomendacoes = priorizar_recomendacoes(candidatas)

# print(f"Total de recomendações: {len(recomendacoes)}")
# display(pd.DataFrame(recomendacoes))

campanha_modificada = aplicar_recomendacoes(campanha, recomendacoes, dados)
# comparacao = pd.DataFrame({"Original": campanha, "Modificada": campanha_modificada})
# display(comparacao)

# Visualizar
display(pd.DataFrame(recomendacoes)[[
    "atributo",
    "valor_atual",
    "valor_recomendado",
    "metrica",
    "diferenca_pct",
    "metrica_principal"
]])

,atributo,valor_atual,valor_recomendado,metrica,diferenca_pct,metrica_principal
0,audio_group,medium_music,high_music,ctr,42.395437,False
1,has_hook,True,False,roas,37.121212,False
2,format,vertical,quadrado,roas,30.985915,False
3,text_density,low,medium,engagement_rate,23.030303,True
4,duration_group,long,short,engagement_rate,17.307692,True


In [32]:
def recommendations_engine(campanha, dados):
    candidatas = gerar_candidatas(campanha, dados)
    recomendacoes = priorizar_recomendacoes(candidatas)

    campanha_modificada = aplicar_recomendacoes(
        campanha, recomendacoes, dados
    )

    return {
        "recomendacoes": recomendacoes,
        "campanha_modificada": campanha_modificada
    }

In [38]:
campanha = dados.iloc[89]
resultado = recommendations_engine(campanha, dados)

display(pd.DataFrame(resultado["recomendacoes"]))

,atributo,valor_atual,valor_recomendado,metrica,diferenca_pct,mediana_atual,mediana_alternativa,n_atual,n_alternativa,metrica_principal
0,has_hook,False,True,ctr,57.474747,0.049500,0.077950,113,128,False
1,has_face,False,True,engagement_rate,54.137931,0.014500,0.022350,114,110,False
2,audio_group,high_music,medium_music,roas,47.058824,1.360000,2.000000,79,71,True
3,duration_group,medium,short,roas,36.981132,1.325000,1.815000,72,78,True
4,format,vertical,quadrado,conversion_rate,26.114359,0.026965,0.034006,106,68,False
